### Парсер

In [ ]:
from lxml import etree
import requests
import pandas as pd
import re
from time import sleep
from dataclasses import dataclass
import numpy as np

In [ ]:
class ArxivParser:
  def __init__(self):
    self.url_template = "https://export.arxiv.org/api/query?search_query=all:";
    self.max_results_template = "&max_results="

  def get_dataset(self, article_topic: str, max_results: int) -> pd.DataFrame:
    root = self.download_data(article_topic, max_results)
    return self.parse_xml(root)

  def download_data(self, article_topic: str, max_results: int) -> etree._Element:
    url = self.url_template + article_topic + self.max_results_template + f"{max_results}"
    response = requests.get(url)
    return etree.fromstring(response.content)

  def parse_xml(self, root: etree._Element) -> pd.DataFrame:
    namespaces = {'atom': 'http://www.w3.org/2005/Atom'}
    entries = root.findall('atom:entry', namespaces=namespaces)

    titles = []
    authors = []
    summaries = []
    links = []

    for entry in entries:
      titles.append(entry.find('atom:title', namespaces).text)
      summaries.append(entry.find('atom:summary', namespaces).text)
      links.append(entry.find('atom:id', namespaces).text)
      names = []
      for name in entry.findall('atom:author/atom:name', namespaces):
        names.append(name.text)
      authors.append('\n'.join(names))

    data = {"title": titles, "summary": summaries, "author" : authors, "link": links}
    return pd.DataFrame(data=data)

Объединяющая все статьи тема: рыба

In [ ]:
theme = "fish"
parser = ArxivParser()
dataset = parser.get_dataset(theme, 500)
dataset

,title,summary,author,link
0,"Designs, Motion Mechanism, Motion Coordination...","In the last few years, there have been many ...",Zhiwei Yu\nKai Li\nYu Ji\nSimon X. Yang,http://arxiv.org/abs/2206.15304v1
1,Hydrodynamics of undulatory fish schooling in ...,The thrust benefits of lateral configuration...,Li Jeany Zhang\nJeff D. Eldredge,http://arxiv.org/abs/1003.4441v1
2,Underwater Fish Detection with Weak Multi-Doma...,"Given a sufficiently large training dataset,...",Dmitry A. Konovalov\nAlzayat Saleh\nMichael Br...,http://arxiv.org/abs/1905.10708v2
3,Absolute 3D Pose Estimation and Length Measure...,Monocular absolute 3D fish pose estimation a...,Jie Mei\nJenq-Neng Hwang\nSuzanne Romain\nCrai...,http://arxiv.org/abs/2102.04639v1
4,Nursery function rehabilitation projects in po...,Conservation measures are implemented to sup...,Etienne Joubert\nCharlotte Sève\nStéphanie Mah...,http://arxiv.org/abs/2406.09470v1
...,...,...,...,...
495,Two-oscillator model of trapped-modes interact...,We discuss the similarity between the nature...,Vladimir R. Tuz\nBogdan A. Kochetov\nLyudmila ...,http://arxiv.org/abs/1408.6486v1
496,Handling oversampling in dynamic networks usin...,Oversampling is a common characteristic of d...,Benjamin Fish\nRajmonda S. Caceres,http://arxiv.org/abs/1504.06667v2
497,Reaction Diffusion patterns in Pseudoplatystom...,This paper studies how patterns derived from...,Aldo Ledesma-Durán\nHéctor Juárez-Valencia\nIv...,http://arxiv.org/abs/1601.00705v1
498,Loose social organisation of AB strain zebrafi...,We explore the collective behaviours of 7 gr...,Axel Séguret\nBertrand Collignon\nLeo Cazenill...,http://arxiv.org/abs/1701.02572v1


### Реализация сплиттера

In [ ]:
class ChunkSplitter:
  def __init__(self, chunk_size=500, overlap=50):
    self.chunk_size = chunk_size
    self.overlap = overlap

  def split(self, text: str) -> list[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), self.chunk_size - self.overlap):
        chunks.append(" ".join(words[i:i + self.chunk_size]))
    return chunks


In [ ]:
splitter = ChunkSplitter(chunk_size=200, overlap=40)
dataset["chunks"] = dataset["summary"].apply(splitter.split)
dataset["chunks"][16][1]

'and dolphin numbers using two function approximation methods: regression and classification. For regression, Densenet201 performed best for fish and Xception best for dolphin with mean squared errors of 2.11 and 0.133 respectively. For classification, InceptionResNetV2 performed best for fish and MobileNetV2 best for dolphins with a mean error of 2.07 and 0.245 respectively. Considering the 123 testing images, our results show the success of data simulation for limited sonar data sets. We find DenseNet201 is able to identify dolphins after approximately 5000 training images, while fish required the full 25,000. Our method can be used to lower costs and expedite the data analysis of fish and dolphin abundance to real-time along the Amazon river and river systems worldwide.'

# Реализация векторного хранилища

In [ ]:
!pip install -qU langchain-community faiss-cpu
import faiss
from langchain_community.vectorstores import FAISS
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

In [ ]:
from langchain_community.docstore.in_memory import InMemoryDocstore
from uuid import uuid4
from langchain_core.documents import Document

class ArticleDatabase:
  def __init__(self):
    self.dataset = None
    self.transformer = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    self.embedding_length = 384
    self.index = None
    self.vector_store = None

  def add_dataset(self, dataset):
    self.dataset = dataset

  def set_vector_store(self):
    self.index = faiss.IndexFlatL2(self.embedding_length)
    self.vector_store = FAISS(
      embedding_function=self.transformer.encode,
      index=self.index,
      docstore=InMemoryDocstore(),
      index_to_docstore_id={},)

    self.fill_vector_store();

  def fill_vector_store(self):
    documents = []
    for index, chunks in enumerate(self.dataset["chunks"]):
        for chunk_index, chunk in enumerate(chunks):
          documents.append(
              Document(
                page_content=chunk,
                metadata={"title": self.dataset["title"][index]}
                )
              )
    uuids = [str(uuid4()) for _ in range(len(documents))]
    self.vector_store.add_documents(documents=documents, ids=uuids)
    return self.vector_store

  def save_database(self, df_path, db_path):
    self.dataset.to_csv(df_path, index=False)
    self.vector_store.save_local(db_path)

  def load_database(self, df_path, db_path):
    self.dataset = pd.read_csv(df_path)
    self.vector_store = FAISS.load_local(db_path, self.transformer.encode,
                                         allow_dangerous_deserialization=True)

  def find_similar_chunks(self, text, number = 1):
    return self.vector_store.similarity_search(text, number)

  def find_similar_articles(self, text, number = 1):
    docs = self.find_similar_chunks(text, number * 3)
    titles = [title for title in [doc.metadata["title"] for doc in docs]]
    return self.dataset[self.dataset["title"].isin(titles)][:number]


Класс создаёт векторное хранилище FAISS и хранит в нём чанки как отдельные документы с метаданными о родительской статье.

При поиске статьи по тексту, мы ищем наиболее подходящие чанки, а затем используя их метаданные(решил хранить title статьи), берём соответсвующие статьи из датасета.

In [ ]:
database = ArticleDatabase()
database.add_dataset(dataset)
database.set_vector_store()

#### Тестирование работы векторного хранилища

In [ ]:
database.find_similar_chunks("Tell me about robotic fish that can swim like real ones", 2)

[Document(id='d2b23742-4696-4e57-b8eb-468790987264', metadata={'title': 'Designs, Motion Mechanism, Motion Coordination, and Communication of\n  Bionic Robot Fishes: A Survey'}, page_content='narrow the gap in swimming performance between robot fishes and fish. This paper is a first step to bring together roboticists and marine biologists interested in learning state-of-the-art research on bionic robot fishes.'),
 Document(id='3b801571-d1ba-4ae2-97ee-950c9be59a80', metadata={'title': 'Designs, Motion Mechanism, Motion Coordination, and Communication of\n  Bionic Robot Fishes: A Survey'}, page_content='In the last few years, there have been many new developments and significant accomplishments in the research of bionic robot fishes. However, in terms of swimming performance, existing bionic robot fishes lag far behind fish, prompting researchers to constantly develop innovative designs of various bionic robot fishes. In this paper, the latest designs of robot fishes are presented in det

In [ ]:
database.find_similar_articles("Tell me about robotic fish that can swim like real ones", 5)

,title,summary,author,link,chunks
0,"Designs, Motion Mechanism, Motion Coordination...","In the last few years, there have been many ...",Zhiwei Yu\nKai Li\nYu Ji\nSimon X. Yang,http://arxiv.org/abs/2206.15304v1,"[In the last few years, there have been many n..."
30,OpenFish: Biomimetic Design of a Soft Robotic ...,We present OpenFish: an open source soft rob...,Sander C. van den Berg\nRob B. N. Scharff\nZol...,http://arxiv.org/abs/2108.12285v2,[We present OpenFish: an open source soft robo...
34,On Locomotion of a Laminated Fish-inspired rob...,Many different robots have been designed and...,Mohammad Sharifzadeh\nRoozbeh Khodambashi\nWen...,http://arxiv.org/abs/1808.09146v1,[Many different robots have been designed and ...
46,The Design and Simulation of Biomimetic Fish R...,In the application of underwater creature st...,Ningzhe Hou,http://arxiv.org/abs/2110.07019v1,[In the application of underwater creature stu...
62,Experimental study of fish-like bodies with pa...,Scombrid fishes and tuna are efficient swimm...,L. Padovani\nG. Manduca\nD. Paniccia\nG. Grazi...,http://arxiv.org/abs/2411.10760v1,[Scombrid fishes and tuna are efficient swimme...


#### Сохранение векторного хранилища

In [ ]:
database.save_database("fish_dataset42", "fish_database42")

# Интеграция с LLM

#### Загрузка векторного хранилища

In [ ]:
!pip install -qU langchain-community faiss-cpu
import faiss
from langchain_community.vectorstores import FAISS
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
loaded_db = ArticleDatabase()
loaded_db.load_database("fish_dataset42", "fish_database42")

In [ ]:
loaded_db.find_similar_articles("I want to write a program that will help me catch fish", 3)

,title,summary,author,link,chunks
19,FishMOT: A Simple and Effective Method for Fis...,Fish tracking plays a vital role in understa...,Shuo Liu\nLulu Han\nXiaoyang Liu\nJunli Ren\nF...,http://arxiv.org/abs/2309.02975v3,['Fish tracking plays a vital role in understa...
35,Fish recognition based on the combination betw...,We presents in this paper a novel fish class...,Mutasem Khalil Sari Alsmadi\nKhairuddin Bin Om...,http://arxiv.org/abs/0912.0986v1,['We presents in this paper a novel fish class...
86,FisherMob : a bioeconomic model of fishers' mi...,"Sea fishing is a highly mobile activity, fav...",Timothée Brochier\nAlassane Bah,http://arxiv.org/abs/2103.16496v1,"[""Sea fishing is a highly mobile activity, fav..."


### Сервис для работы с моделью

Сервис интегрирован с моделью flan-t5-large.

Реализован следующий функционал:

1. ask_ai_about: Пользователь задаёт запрос, из базы данных берётся 5 самых подходящих чанков, для каждого чанка LLM генерирует ответ на вопрос, а потом резюмирует предыдущие ответы в один

2. get_main_ideas: Пользователь задаёт запрос, из базы данных берётся самая подходящая статья (то есть статья, которой соответсвует самый подходящий чанк). Потом чанки статьи передаются LLM с промтом на выделение ключевых идей. Затем набор из выделенных идей LLM объединяет в один ответ.

3. find_articles: Находит топ n статей лучше всего подходящих под запрос, flan не используется.

Все пользовательские запросы автоисправляет speller SymSpell

In [ ]:
!pip install symspellpy
import pkg_resources
from transformers import T5Tokenizer, T5ForConditionalGeneration
from symspellpy.symspellpy import SymSpell, Verbosity

class ArticleService:
  def __init__(self, db: ArticleDatabase):
    self.db = db
    self.tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
    self.model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")
    self.chunk_count = 5
    self.speller = None
    self.set_speller()

  def send_query(self, query: str):
    input_ids = self.tokenizer(query, return_tensors="pt").input_ids
    output = self.model.generate(input_ids)
    return self.tokenizer.decode(output[0]).replace("<pad>", "").replace("</s>", "")

  def ask_ai_about(self, question: str):
    question = self.сorrect_query(question)
    similar_chunks = self.db.find_similar_chunks(question, self.chunk_count)
    answers = []
    for i, chunk in enumerate(similar_chunks):
      content = chunk.page_content
      answer = self.send_query(self.get_question_prompt(content, question))
      answers.append(answer)
      print(f"Chunk#{i} answer:{answer}")
    resume_prompt = self.get_resume_promt("\n".join(answers), question)
    return self.send_query(resume_prompt)

  def get_main_ideas(self, article_description:str):
    article_description = self.сorrect_query(article_description)
    similar_article = self.db.find_similar_articles(article_description, 1)
    print(f"Article title: {similar_article['title'].values[0]}")
    answers = []
    for i, chunk in enumerate(similar_article["chunks"].values):
      answer = self.send_query(self.get_main_ideas_promt(chunk))
      answers.append(answer)
      print(f"Chunk#{i} ideas: {answer}")
    resume_prompt = self.get_main_resume_promt("\n".join(answers),)
    return self.send_query(resume_prompt)

  def find_articles(self, article_description:str, count = 5):
    article_description = self.сorrect_query(article_description)
    similar_article = self.db.find_similar_articles(article_description, count)
    return similar_article

  def set_speller(self):
    self.speller = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
    dictionary = pkg_resources.resource_filename("symspellpy", "frequency_dictionary_en_82_765.txt")
    self.speller.load_dictionary(dictionary, term_index=0, count_index=1)

  def сorrect_query(self, text):
    suggestions = self.speller.lookup_compound(text, max_edit_distance=2)
    corrected_query = suggestions[0].term
    print(f"Speller suggest: {corrected_query}")
    return corrected_query

  def get_question_prompt(self, context: str, user_query: str):
    prompt = f'''
    You are an AI assistant that answers questions based on research papers.
Use the given context from article to answer accurately.

### Context:
{context}

### User Question:
{user_query}
'''
    return prompt

  def get_resume_promt(self, context:str, user_query: str):
    prompt = f'''
    You are an AI assistant that answers questions based on research papers.
Summarize the following answers from context into a single sentence.

Context: {context}

User Question: {user_query}
'''
    return prompt

  def get_main_ideas_promt(self, context:str):
    prompt = f'''
    You are an AI assistant that answers questions based on research papers.
Highlight the main ideas from the article сontext.

Context: {context}
'''
    return prompt

  def get_main_resume_promt(self, context:str):
    prompt = f'''
    You are an AI assistant that answers questions based on research papers.
Summarize the following ideas from context into a single sentence.

Context: {context}
'''
    return prompt

In [ ]:
articleService = ArticleService(loaded_db)

### Тестирование запросов

In [ ]:
print(f"Resume: {articleService.ask_ai_about('What tools do people use when examining fish?')}")

Speller suggest: what tools do people use when examining fish
Chunk#0 answer: devices
Chunk#1 answer: biochemical composition
Chunk#2 answer: existing datasets
Chunk#3 answer: Regular cameras are combined with event cameras and a scanning sonar
Chunk#4 answer: unwind
Resume:  Regular cameras are combined with event cameras and a scanning sonar unwind


In [ ]:
print(f"Resume: {articleService.ask_ai_about('best machine learning models')}")

Speller suggest: best machine learning models
Chunk#0 answer: Random Forest
Chunk#1 answer: MobileNet
Chunk#2 answer:other two datasets that were included in the training set were higher than the scores achieved by the model
Chunk#3 answer:Random Forests predict and optimize water temperature and pH levels for the fish, fostering their health and
Chunk#4 answer: DeepSalmon
Resume: Random Forest MobileNet other two datasets that were included in the training set were higher than the scores


In [ ]:
articleService.get_main_ideas("article about computer vision algorithms")

Speller suggest: article about computer vision algorithms
Article title: Automatic Controlling Fish Feeding Machine using Feature Extraction of
  Nutriment and Ripple Behavior
Chunk#0 ideas:To build robust method for reasonable application, we propose automatic controlling fish feeding machine based on computer vision


'To build robust method for reasonable application, we propose automatic controlling fish feeding machine based on computer vision'

In [ ]:
articleService.get_main_ideas("article about fish behavior")

Speller suggest: article about fish behavior
Article title: Fishlike Rheotaxis
Chunk#0 ideas: We propose a theoretical model consisting of a fishlike body equipped with lateral pressure sensors


'We propose a theoretical model consisting of a fishlike body equipped with lateral pressure sensors'

In [ ]:
articleService.get_main_ideas("research of bionic robot fishes")

Speller suggest: research of bionic robot fishes
Article title: Designs, Motion Mechanism, Motion Coordination, and Communication of
  Bionic Robot Fishes: A Survey
Chunk#0 ideas: In the last few years, there have been many new developments and significant accomplishments in the research of


'In the last few years, there have been many new developments and significant accomplishments in the research of'

In [ ]:
articleService.find_articles("articles about machine learning algorithms", count=7)

Speller suggest: articles about machine learning algorithms


,title,summary,author,link,chunks
25,IoT-Based Environmental Control System for Fis...,In response to the burgeoning global demand ...,D. Dhinakaran\nS. Gopalakrishnan\nM. D. Maniga...,http://arxiv.org/abs/2311.04258v1,"[""In response to the burgeoning global demand ..."
39,Lightweight Fish Classification Model for Sust...,The enormous demand for seafood products has...,Febrian Kurniawan\nGandeva Bayu Satrya\nFiruz ...,http://arxiv.org/abs/2401.02278v1,['The enormous demand for seafood products has...
78,Fish Disease Detection Using Image Based Machi...,Fish diseases in aquaculture constitute a si...,Md Shoaib Ahmed\nTanjim Taharat Aurpa\nMd. Abu...,http://arxiv.org/abs/2105.03934v1,['Fish diseases in aquaculture constitute a si...
112,A Machine Learning Approach for Early Detectio...,Early detection of fish diseases and identif...,Al-Akhir Nayan\nAhamad Nokib Mozumder\nJoyeta ...,http://arxiv.org/abs/2102.09390v2,['Early detection of fish diseases and identif...
131,The Fishnet Open Images Database: A Dataset fo...,Camera-based electronic monitoring (EM) syst...,Justin Kay\nMatt Merrifield,http://arxiv.org/abs/2106.09178v1,['Camera-based electronic monitoring (EM) syst...
166,Research on Optimization Method of Multi-scale...,The fish target detection algorithm lacks a ...,Yang Liu\nShengmao Zhang\nFei Wang\nWei Fan\nG...,http://arxiv.org/abs/2104.05050v1,['The fish target detection algorithm lacks a ...
225,Optimal Quota for a Multi-species Fishing Models,A Stochastic Control Problem can be solved b...,Olivier Pironneau,http://arxiv.org/abs/2309.06253v1,['A Stochastic Control Problem can be solved b...


In [ ]:
print(f"Resume: {articleService.ask_ai_about('Hgw petople cin casch fihh in sea?')}")

Speller suggest: how people in cash fish in ser
Chunk#0 answer:
Chunk#1 answer: artificial structures
Chunk#2 answer: pelagic pair trawlers
Chunk#3 answer:with continued observations, and the appropriate covariates, we should ultimately be able to uncover the
Chunk#4 answer: A fish farm is an area where fish raise and bred for food
Resume: artificial structures pelagic pair trawlers with continued observations, and the appropriate covari


In [ ]:
articleService.find_articles('Hgw petople can catch fihh in sea?', 2)

Speller suggest: how people can catch fish in sea


,title,summary,author,link,chunks
4,Nursery function rehabilitation projects in po...,Conservation measures are implemented to sup...,Etienne Joubert\nCharlotte Sève\nStéphanie Mah...,http://arxiv.org/abs/2406.09470v1,['Conservation measures are implemented to sup...
163,Under Water Waste Cleaning by Mobile Edge Comp...,As water pollution is a serious threat to un...,Subhadeep Sahoo\nXiao Han Dong\nZi Qian Liu\nJ...,http://arxiv.org/abs/2009.00072v1,['As water pollution is a serious threat to un...
